# Portion 3 · Baselines and supervised experiments

Runs the tabular half of Portion 3 through `Portion3/portion3_baselines.py`: the rules
baseline, Isolation Forest, LightGBM on engineered features, and LightGBM plus the frozen
Portion 2 embeddings. Everything here is CPU-only and runs on a laptop. The fine-tuning
grid is in `Portion3_Finetune.ipynb`.

- Full runbook: `Portion3/HANDOFF.md`
- Run matrix: `Portion3/RUN_PLAN.md`
- Never edit `results/runs.csv` by hand; every command appends to it.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# The notebook lives in Portion3/, so the repository root is one level up.
REPO = Path.cwd()
if not (REPO / "Portion3").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

PROCESSED = REPO / "Portion1" / "processed"
EMB_DIR = Path(os.environ.get("TABBERT_EMBEDDINGS", REPO / "Portion3" / "outputs" / "embeddings_256"))
RESULTS = REPO / "results" / "runs.csv"
OUTPUTS = REPO / "Portion3" / "outputs"
assert PROCESSED.exists(), f"missing {PROCESSED}"
print("repo:", REPO)
print("embeddings:", EMB_DIR, "(exists:", EMB_DIR.exists(), ")")

def run_cli(*args):
    # Run the Portion 3 command line with this kernel's interpreter.
    cmd = [sys.executable, "-m", "Portion3.portion3_baselines", *map(str, args)]
    print("$", " ".join(cmd[2:]))
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0:
        raise RuntimeError("command failed")

## 1. Check the plan

60 runs remain: 40 tabular (this notebook) and 20 fine-tuning (the other notebook).
`status` comes from `results/runs.csv`, so rerunning is always safe.

In [ ]:
run_cli("plan")

## 2. Baselines on engineered features

No embeddings needed for this section. Rules take under a minute; Isolation Forest about
5 minutes per seed; LightGBM about 5 to 15 minutes per fraction across three seeds.

In [ ]:
run_cli("rules")
run_cli("isolation-forest")
run_cli("lightgbm")

## 3. Light hyperparameter check (validation only, needs embeddings)

Compares `num_leaves` 31 vs 63 and 256 vs 512 embedding columns at 5% labels, seed 42,
scored on validation PR AUC. It writes `Portion3/configs/lightgbm_tuning.json`; update
`Portion3/configs/lightgbm.json` with the recommended values before the next step.
Skip this section if you are happy with the shipped defaults (31 leaves, 256 columns).

In [ ]:
if EMB_DIR.exists():
    run_cli("tune", "--embeddings-dir", EMB_DIR)
else:
    print("embeddings not found:", EMB_DIR)
    print("run Portion3_PrepareEmbeddings.ipynb (Colab) or prepare_embeddings slice (local) first")

## 4. LightGBM plus frozen embeddings

Pairs downstream seed s with `main_seed<s>` and repeats Experiment C with the two ablation
embeddings at 5% labels. Set `TABBERT_EMBEDDINGS` or edit `EMB_DIR` above.

In [ ]:
run_cli("lightgbm-embeddings", "--embeddings-dir", EMB_DIR)

## 5. Results

`summary` prints mean and spread of test PR AUC by method and fraction. The full log is
`results/runs.csv`; commit it when the stage is complete.

In [ ]:
run_cli("summary")

import pandas as pd
frame = pd.read_csv(RESULTS)
pd.set_option("display.width", 200)
print(frame[["run_id", "method", "label_fraction", "seed", "valid_ap", "test_ap", "test_f1"]].to_string(index=False))